# 🏛️ La Prueba Final ante el Consejo de Ancianos (RAG Avanzado)
> *Los Ancianos observan desde las sombras de la cámara sagrada: "Demuestra que tu sistema no solo recupera fragmentos, sino que juzga la relevancia suprema, filtra la verdad y evalúa la Tríada sin margen de error."*

**Objetivo:** Implementar Re-Ranking con Cross-Encoders, filtrado por metadatos y evaluación cuantitativa de la Tríada RAG.

In [ ]:
# 🥷 Carga de librerías avanzadas y Cross-Encoder
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from sentence_transformers import CrossEncoder

print("✅ Artefactos de nivel maestro cargados.")

### 📜 Paso 1: Ingesta y Fragmentación con Metadatos
> *El Sabio aconseja: "Añade sellos de metadatos a cada fragmento para que el Consejo pueda filtrar la verdad por origen y sección."*

In [ ]:
# 🥷 Ingesta y asignación de metadatos
loader = PyPDFLoader("docs/teoria_rag.pdf")
docs = loader.load()

# Asignar metadatos personalizados a cada página
for i, doc in enumerate(docs):
    doc.metadata["capitulo"] = f"Capitulo_{i+1}"
    doc.metadata["nivel_acceso"] = "Ancianos"

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(),
    collection_name="santuario_avanzado"
)
print("🧩 Fragmentos cargados y etiquetados con metadatos.")

### ⚖️ Paso 2: El Re-Ranking del Cross-Encoder
> *"La búsqueda vectorial por similitud de coseno a veces recupera ruido. El Cross-Encoder reordena los fragmentos analizando la relación profunda entre pregunta y contexto."*

In [ ]:
# 🥷 Implementación del modelo de Re-Ranking
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank_documents(query, retrieved_docs, top_n=2):
    pairs = [[query, doc.page_content] for doc in retrieved_docs]
    scores = cross_encoder.predict(pairs)

    # Ordenar documentos según el puntaje del Cross-Encoder
    scored_docs = sorted(zip(scores, retrieved_docs), key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored_docs[:top_n]]

# Prueba de recuperación amplia + Re-ranking
query = "¿Cómo evita la Tríada RAG las alucinaciones del modelo?"
initial_docs = vectorstore.similarity_search(query, k=5)
ranked_docs = rerank_documents(query, initial_docs, top_n=2)

print(f"🔍 Iniciales: {len(initial_docs)} ➔ Re-ordenados (Top 2): {len(ranked_docs)}")

### 🛡️ Paso 3: Filtrado por Metadatos y Generación
> *"Solo permitimos que el oráculo consulte información clasificada con el metadato correcto."*

In [ ]:
# 🥷 Recuperación filtrada + Sintesis restrictiva
filtered_docs = vectorstore.similarity_search(
    query,
    k=3,
    filter={"nivel_acceso": "Ancianos"}
)

context_text = "\n\n".join([d.page_content for d in filtered_docs])

system_prompt = f"""Eres el Sabio. Responde a la pregunta únicamente con este contexto validado:
{context_text}

Si la respuesta no se deduce del contexto, responde: 'Información rechazada por el Consejo.'"""

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
respuesta = llm.invoke([("system", system_prompt), ("human", query)])

print("🤖 Respuesta ante el Consejo:\n")
print(respuesta.content)

### 📐 Paso 4: Evaluación Cuantitativa de la Tríada (RAG Triad)
> *El Consejo de Ancianos exige métricas explícitas: Relevancia del Contexto, Fidelidad (Groundedness) y Relevancia de la Respuesta.*

In [ ]:
# 🥷 Módulo de evaluación heurística de la Tríada
eval_prompt = f"""Evalúa las siguientes 3 métricas de 0.0 a 1.0 para la interacción:
Pregunta: {query}
Contexto: {context_text}
Respuesta: {respuesta.content}

Responde en formato JSON estrictamente con las llaves:
'context_relevance', 'groundedness', 'answer_relevance'"""

eval_result = llm.invoke([("system", "Responde solo en formato JSON válido."), ("human", eval_prompt)])
print("📊 Evaluación del Consejo (Métricas RAG):\n")
print(eval_result.content)

### 📝 Veredicto del Consejo de Ancianos
> *Los Ancianos se inclinan con respeto: "Has dominado el Re-ranking, los metadatos y la evaluación cuantitativa. El Gran Archivo está seguro bajo tu tutela."*

### 🎯 Resumen del Nivel Avanzado

1. **Re-Ranking:** Resuelve el problema de falsos positivos en búsquedas semánticas primarias mediante `CrossEncoder`.
2. **Metadata Filtering:** Permite segmentar el espacio de búsqueda vectorizado según niveles de acceso o categorías.
3. **Métricas de Evaluación:** Implementación del patrón RAG Triad para auditoría automatizada de producción.